**НИУ ВШЭ · программа ФТиАД · магистратура, 1-й курс**

# Домашнее задание № 1 по эконометрике

## Сравнение цен домов с учётом площади и района

Сравните цены домов двух типов с учётом площади. С помощью трёх моделей выясните, как изменится оценка разницы в ценах, если учесть район.

### Что сдать и что принести на лекцию

- **До 23:59 15 ноября** отправьте файл `.ipynb` с кодом, результатами расчётов и ответами на [aj.zhimerikina@physics.msu.ru](mailto:aj.zhimerikina@physics.msu.ru). Записывайте выводы рядом с расчётами. За каждое из шести заданий можно получить 1 балл.
- На лекции **16 ноября в 18:10** вы получите три вопроса по результатам своей модели 3. На ответы отводится 30 минут. Принесите заполненный лист с последней страницы файла «Домашнее_задание_01.pdf». Во время работы можно пользоваться **только распечатанным листом с результатами**. За ответы можно получить до 4 баллов.

### Данные и номер варианта

В данных представлены сделки купли-продажи домов с одним, двумя или тремя жилыми помещениями в пяти районах Нью-Йорка с октября 2025 по сентябрь 2026 года. Каждая строка описывает одну сделку.

Файл `nyc_homes_2025-10_2026-09.csv.gz` лежит в папке `data` рядом с ноутбуком. Данные выгружены 18 октября 2026 года с сайта Департамента финансов Нью-Йорка. Источник и правила отбора описаны в `data/source_info.json`.

Укажите свой номер варианта (от 1 до 60) в `VARIANT` и запустите первую ячейку с кодом. Она отберёт 3 000 наблюдений и выведет таблицу `target` с характеристиками дома для прогноза.


In [1]:
# При необходимости установите зависимости и перезапустите ядро:
%pip install -r requirements.txt

  Using cached patsy-1.0.3-py2.py3-none-any.whl.metadata (3.7 kB)
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   --- ------------------------------------ 1.0/12.6 MB 4.6 MB/s eta 0:00:03
   --- ------------------------------------ 1.0/12.6 MB 4.6 MB/s eta 0:00:03
   --- ------------------------------------ 1.0/12.6 MB 4.6 MB/s eta 0:00:03
   ---- ----------------------------------- 1.3/12.6 MB 1.4 MB/s eta 0:00:08
   ----- ---------------------------------- 1.6/12.6 MB 1.3 MB/s eta 0:00:09
   ----- ---------------------------------- 1.6/12.6 MB 1.3 MB/s eta 0:00:09
   ----- ---------------------------------- 1.8/12.6 MB 1.2 MB/s eta 0:00:09
   ------ --------------------------------- 2.1/12.6 MB 1.2 MB/s eta 0:00:09
   ------- -------------------------------- 2.4/12.6 MB 1.2 MB/s eta 0:00:09
   -------- ------------------------------- 2.6/12.6 MB 1.2 MB/s eta 0:00:09
   --------- ------------------------------ 2.9/12.6 MB 1.2 MB/s eta 0:00:09
   ---------- ---

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels.api as sm

VARIANT = 12  # Замените None на свой номер варианта от 1 до 60.
if type(VARIANT) is not int or not 1 <= VARIANT <= 60:
    raise ValueError('Замените None на номер варианта от 1 до 60.')
data_path = Path('data') / 'nyc_homes_2025-10_2026-09.csv.gz'
if not data_path.is_file():
    raise FileNotFoundError('Папка data с файлом nyc_homes_2025-10_2026-09.csv.gz должна лежать рядом с ноутбуком.')
base = pd.read_csv(data_path)
data = base.sample(n=3000, random_state=20261019 + VARIANT).copy()
target = pd.DataFrame([{
    'area_m2': [140, 160, 180, 200, 220][(VARIANT - 1) % 5],
    'multi_family': ((VARIANT - 1) // 5) % 2,
    'borough': [2, 3, 4, 5][(VARIANT - 1) % 4],
}])
print('Вариант:', VARIANT, '| Число наблюдений:', len(data))
print('Характеристики дома для прогноза:')
display(target)
data[['price', 'area_m2', 'multi_family', 'borough']].head()


Вариант: 12 | Число наблюдений: 3000
Характеристики дома для прогноза:


,area_m2,multi_family,borough
0,160,0,5


,price,area_m2,multi_family,borough
15548,532.481,193.238323,0,3
7256,910.000,110.368812,0,4
5446,900.000,100.335283,0,4
10395,735.000,131.922317,0,5
11366,1500.000,316.056142,1,2


### Описание данных

| Колонка | Обозначение | Единицы или коды | Что означает |
| --- | --- | --- | --- |
| `price` | $P$ | тыс. долл. США | Цена всего дома |
| `area_m2` | $S$ | м² | Площадь здания |
| `multi_family` | $M$ | 0 или 1 | $M=0$ для одного жилого помещения, $M=1$ для двух или трёх |
| `borough` | Район | от 1 до 5 | Manhattan (1), Bronx (2), Brooklyn (3), Queens (4), Staten Island (5) |

### Три модели

$$
\begin{aligned}
\text{Модель 1.}\quad & \ln(P)=\beta_0+\beta_1z+\beta_2M+\varepsilon \\
\text{Модель 2.}\quad & \ln(P)=\beta_0+\beta_1z+\beta_2M+G+\varepsilon \\
\text{Модель 3.}\quad & \ln(P)=\beta_0+\beta_1z+\beta_2M+G+\delta Mz+\varepsilon \\
& G=\alpha_1D_1+\alpha_2D_2+\alpha_3D_3+\alpha_5D_5 .
\end{aligned}
$$

Модель 1 учитывает площадь и тип дома. Модель 2 добавляет район. В модели 3 наклон может различаться для двух типов домов. Коэффициенты оценивайте отдельно для каждой модели.

Во всех моделях зависимая переменная — натуральный логарифм цены $\ln(P)$ по основанию $e$. Обозначим $z=\ln(S/150)$. Базовый район — Queens (код 4). Индикатор $D_j$ равен 1 для домов в районе $j$ и 0 для остальных.

### Предпосылки модели

Считайте, что при заданных значениях признаков ошибки независимы и нормально распределены с нулевым средним и одинаковой дисперсией. Используйте эти предпосылки при проверке гипотез и построении интервалов.

Модели оценивают средний логарифм цены при заданных признаках. Под прогнозом цены в задании понимаем экспоненту прогноза логарифма. При указанных предпосылках это оценка медианы цены для домов с такими признаками.


## 1. Проверьте выборку и запишите ожидания

1. Проверьте, что в вашей выборке 3 000 наблюдений.
2. Посчитайте число сделок по районам и типам домов. В строках таблицы укажите районы, в столбцах — типы домов. Проверьте, представлены ли оба типа в каждом районе.
3. До оценки моделей запишите и обоснуйте два предположения в 3–4 предложениях. Как, по-вашему, цена связана с площадью? Как может измениться оценка разницы в ценах двух типов домов, если учесть район?

In [ ]:
# Для таблицы подойдёт pd.crosstab с borough в строках и multi_family в столбцах.

# Ваш код


**Итоги проверки выборки и ожидания до оценки моделей**



## 2. Оцените три модели

1. Оцените модели 1–3 по отдельности методом наименьших квадратов на одной и той же выборке из 3 000 наблюдений.
2. Для каждой модели укажите число наблюдений и R² для ln(P).
3. Сведите в таблицу оценки коэффициентов, их стандартные ошибки и 95%-е доверительные интервалы.

In [ ]:
# Включите константу во все модели. В моделях с районными индикаторами базовый район — Queens.

# Ваш код


## 3. Сравните цены

Для каждого прогноза подготовьте значения признаков, включая z и, в модели 3, Mz. Получите прогнозы логарифма цены через predict или подстановкой в уравнение модели. Возьмите экспоненту от каждого прогноза и посчитайте, на сколько процентов прогноз цены отличается от базового. В ноутбуке укажите значения признаков и покажите расчёт. Округляйте до двух знаков после запятой только итоговые проценты.

1. Выберите исходную площадь S. По модели 2 сравните прогноз для площади 1,10S с прогнозом для S, сохраняя тип дома и район.
2. Выберите одну площадь. По каждой из моделей 1 и 2 сравните прогноз для M = 1 с прогнозом для M = 0. Возьмите M = 0 за базу. Для модели 2 используйте один район в обоих прогнозах.
3. По модели 3 повторите сравнение типов домов при 150 м² и при 200 м². Используйте один район для всех прогнозов.
4. Как учёт района меняет оценку разницы между типами домов в моделях 1 и 2? Как эта разница меняется с площадью в модели 3?

In [ ]:
# 3.1. Увеличение площади на 10%, модель 2
# Подготовьте отдельную строку с признаками для каждого прогноза.

# Ваш код


In [ ]:
# 3.2. Сравнение типов домов, модели 1 и 2

# Ваш код


In [ ]:
# 3.3. Сравнение типов домов при 150 м² и 200 м², модель 3

# Ваш код


**Сравнение результатов (пункт 3.4)**



## 4. Проверьте равенство наклонов

По модели 3 проверьте, одинаков ли наклон зависимости ln(P) от z для двух типов домов.

1. Перед расчётами сформулируйте нулевую и альтернативную гипотезы через коэффициент δ при Mz.
2. Проведите двусторонний t-тест для δ или F-тест для сравнения моделей 2 и 3.
3. Укажите значение статистики теста, её распределение при H₀ с числом степеней свободы и p-value.
4. Примите решение при уровне значимости 5%. Скажите, дают ли данные основания считать наклоны разными.

**Запишите гипотезы перед расчётами**

- $H_0$: …
- $H_1$: …

In [ ]:
# Для F-теста укажите степени свободы числителя и знаменателя.

# Ваш код


**Результаты теста и вывод о наклонах**

- Выбранный тест: …
- Значение статистики: …
- Распределение статистики при $H_0$: …
- Степени свободы: …
- p-value: …
- Решение при уровне значимости 5% и вывод о наклонах: …


## 5. Оцените медиану цены и постройте интервалы

1. Выпишите из таблицы target в ноутбуке площадь, тип и район дома для прогноза.
2. По модели 3 получите прогноз логарифма цены и 95%-й доверительный интервал для среднего логарифма. Возьмите экспоненту от прогноза и от обеих границ. Так вы получите оценку медианы цены и доверительный интервал для неё.
3. Постройте 95%-й прогнозный интервал для цены одного нового дома с теми же характеристиками. Сначала найдите границы на шкале логарифма, затем возьмите экспоненту от каждой. Объясните, почему прогнозный интервал шире доверительного.

Запишите оценку медианы и границы обоих интервалов в тысячах долларов.

In [ ]:
# В statsmodels границы двух интервалов можно получить через get_prediction().summary_frame().
# Если передаёте матрицу признаков, включите константу и сохраните порядок колонок модели 3.
# При вызове sm.add_constant для одной строки укажите has_constant='add'.

# Ваш код


**Характеристики дома и объяснение разницы между интервалами**



## 6. Подведите итог и заполните лист с результатами

1. Напишите в ноутбуке общий вывод в 5–7 предложениях. Опишите, как учёт района изменил оценку разницы в ценах двух типов домов и совпало ли это с вашими ожиданиями. Выберите модель для описания этой разницы, опираясь на численные результаты заданий 3 и 4. Назовите одно ограничение анализа.
2. Заполните лист с результатами модели 3 на последней странице PDF.

**Общий вывод**

